# Fetal Health Classification
### Progetto di Machine Learning — Start2Impact

**Obiettivo:** Costruire un modello predittivo capace di classificare lo stato di salute fetale in tre categorie:
- **1 = Normal**
- **2 = Suspect**
- **3 = Pathological**

I dati provengono da cardiotocogrammi (CTG), esami prenatali che misurano la frequenza cardiaca fetale e le contrazioni uterine.

## 0. Setup e Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, accuracy_score, balanced_accuracy_score
)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

RANDOM_STATE = 42
print('Import completati con successo.')

## 1. Caricamento del Dataset e Train/Test Split

**Regola fondamentale:** il test set viene messo da parte subito e non viene toccato fino alla valutazione finale. Tutta l'analisi esplorativa, la standardizzazione, la selezione delle feature e il tuning degli iperparametri avvengono esclusivamente sul training set. Questo evita il **data leakage**.

In [ ]:
df = pd.read_csv('fetal_health.csv')

print(f'Shape del dataset completo: {df.shape}')
print(f'\nColonne: {list(df.columns)}')
print(f'\nTipo dei dati:\n{df.dtypes}')

In [ ]:
X = df.drop('fetal_health', axis=1)
y = df['fetal_health'].astype(int)  # converto in int: 1, 2, 3

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set: {X_train.shape[0]} campioni')
print(f'Test set:     {X_test.shape[0]} campioni')
print(f'\nDistribuzione classi nel training set:')
print(y_train.value_counts().sort_index())
print(f'\nDistribuzione classi nel test set:')
print(y_test.value_counts().sort_index())

Ho usato `stratify=y` per garantire che la proporzione delle tre classi sia mantenuta sia nel training che nel test set. Questo è fondamentale con dataset sbilanciati.

## 2. Exploratory Data Analysis (EDA)

### 2.1 Statistiche descrittive (training set)

In [ ]:
X_train.describe().round(2)

### 2.2 Valori mancanti

In [ ]:
missing = X_train.isnull().sum()
print('Valori mancanti per feature:')
print(missing[missing > 0] if missing.sum() > 0 else 'Nessun valore mancante — il dataset è completo.')

**Commento:** Il dataset non presenta valori mancanti. Non è necessario applicare alcuna strategia di imputation.

### 2.3 Bilanciamento del dataset

In [ ]:
counts = y_train.value_counts().sort_index()
labels = ['Normal (1)', 'Suspect (2)', 'Pathological (3)']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, counts.values, color=['steelblue', 'orange', 'crimson'], edgecolor='black')
for bar, count in zip(bars, counts.values):
    pct = count / len(y_train) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{count}\n({pct:.1f}%)', ha='center', fontsize=11)
ax.set_title('Distribuzione delle classi (Training Set)', fontsize=14)
ax.set_ylabel('Numero di campioni')
ax.set_ylim(0, max(counts.values) * 1.2)
plt.tight_layout()
plt.show()

print(f'\nRapporto classe maggioritaria/minoritaria: {counts.max()/counts.min():.1f}x')

**Commento sul bilanciamento:** Il dataset è fortemente sbilanciato:
- Normal (1): ~78%
- Suspect (2): ~14%
- Pathological (3): ~8%

**Perché non applico oversampling/undersampling?**
Con soli 2126 campioni totali, tecniche come SMOTE o undersampling andrebbero a ridurre ulteriormente un dataset già piccolo, o a generare campioni sintetici di dubbia qualità per un problema medico. Preferisco gestire lo sbilanciamento a livello di:
1. **Metrica di valutazione** (F1-macro invece di accuracy)
2. **Parametro `class_weight='balanced'`** nei modelli che lo supportano
3. **StratifiedKFold** nella cross-validation

### 2.4 Analisi degli outlier

In [ ]:
# Calcolo outlier con metodo IQR per feature
outlier_info = {}
for col in X_train.columns:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((X_train[col] < lower) | (X_train[col] > upper)).sum()
    pct = n_out / len(X_train) * 100
    outlier_info[col] = {'n_outliers': n_out, 'pct': round(pct, 2)}

outlier_df = pd.DataFrame(outlier_info).T.sort_values('pct', ascending=False)
print('Outlier per feature (metodo IQR):')
print(outlier_df[outlier_df['n_outliers'] > 0])

In [ ]:
# Boxplot di tutte le feature
cols = X_train.columns.tolist()
n_cols_grid = 4
n_rows_grid = int(np.ceil(len(cols) / n_cols_grid))

fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(20, n_rows_grid * 4))
axes = axes.flatten()

for i, col in enumerate(cols):
    axes[i].boxplot(X_train[col].dropna())
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(axis='x', which='both', bottom=False, labelbottom=False)

for j in range(len(cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplot delle feature — outlier visibili come punti fuori dai baffi', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Feature values vs classes (per le feature con più outlier)
top_features = outlier_df.head(6).index.tolist()
class_labels = {1: 'Normal', 2: 'Suspect', 3: 'Pathological'}
colors = {1: 'steelblue', 2: 'orange', 3: 'crimson'}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# Uso reset_index per allineare gli indici
X_train_ri = X_train.reset_index(drop=True)
y_train_ri  = y_train.reset_index(drop=True)

for i, col in enumerate(top_features):
    for cls in [1, 2, 3]:
        mask = y_train_ri == cls
        subset = X_train_ri.loc[mask, col]
        axes[i].scatter(
            [cls] * len(subset),
            subset.values,
            alpha=0.3, s=15, color=colors[cls], label=class_labels[cls]
        )
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xticks([1, 2, 3])
    axes[i].set_xticklabels(['Normal', 'Suspect', 'Pathological'], fontsize=9)

plt.suptitle('Feature values vs Classes (feature con piu outlier)', fontsize=13)
plt.tight_layout()
plt.show()

**Commento sugli outlier:** Sono presenti outlier in diverse feature, soprattutto in quelle legate alle decelerazioni e alla variabilità. Tuttavia:
1. In un contesto medico, i valori estremi possono essere **clinicamente significativi** — un feto con frequenza cardiaca molto bassa o molto alta è proprio il caso che vogliamo identificare.
2. I valori estremi tendono a concentrarsi nelle classi Suspect e Pathological, il che li rende informativi.
3. Non li rimuovo: mantengo tutti i campioni e affronto la variabilità tramite la StandardScaler.

### 2.5 Standardizzazione (StandardScaler)

Le feature hanno scale molto diverse (es. `baseline value` ~120, `accelerations` ~0.003). Quasi tutti gli algoritmi di ML (specialmente KNN e Logistic Regression) sono sensibili alle differenti scale. Applico la StandardScaler **esclusivamente sul training set** e poi la utilizzo per trasformare anche il test set, evitando data leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Ritorno a DataFrame per comodità
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X_test.columns)

print('Statistiche dopo StandardScaler (training):')
print(X_train_scaled.describe().round(3).loc[['mean','std']].T.head(5))
print('\n→ Media ≈ 0, Std ≈ 1 per tutte le feature.')

### 2.6 Matrice di correlazione

In [ ]:
corr_matrix = X_train_scaled.corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Matrice di correlazione (triangolo inferiore)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Coppie con correlazione assoluta > 0.8
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        c = corr_matrix.iloc[i, j]
        if abs(c) > 0.8:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], round(c, 3)))

print('Coppie con |correlazione| > 0.8:')
for a, b, c in sorted(high_corr, key=lambda x: -abs(x[2])):
    print(f'  {a}  <-->  {b}:  {c}')

In [ ]:
# Plot delle coppie altamente correlate
if high_corr:
    n_pairs = min(len(high_corr), 4)
    fig, axes = plt.subplots(1, n_pairs, figsize=(5*n_pairs, 5))
    if n_pairs == 1:
        axes = [axes]
    
    color_map = {1: 'steelblue', 2: 'orange', 3: 'crimson'}
    y_train_arr = y_train.values  # array posizionale, allineato con X_train_scaled
    
    for idx, (a, b, c) in enumerate(high_corr[:n_pairs]):
        for cls in [1, 2, 3]:
            mask = y_train_arr == cls
            axes[idx].scatter(
                X_train_scaled.loc[:, a].values[mask],
                X_train_scaled.loc[:, b].values[mask],
                alpha=0.4, s=15, color=color_map[cls],
                label=class_labels[cls]
            )
        axes[idx].set_xlabel(a, fontsize=9)
        axes[idx].set_ylabel(b, fontsize=9)
        axes[idx].set_title(f'r = {c}', fontsize=11)
        axes[idx].legend(fontsize=8)
    
    plt.suptitle('Feature altamente correlate', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Nessuna coppia con |correlazione| > 0.8 trovata.')

**Commento sulla correlazione:** Le coppie ad alta correlazione (es. `histogram_mean` / `histogram_median` / `histogram_mode`) sono ridondanti. Tuttavia, invece di rimuoverle manualmente, delegherò questo processo alla **feature selection** nel passaggio successivo — che userà l'informazione statistica per scegliere le migliori feature in modo oggettivo.

## 3. Feature Selection

### 3.1 Mutual Information

In [ ]:
mi_scores = mutual_info_classif(X_train_scaled, y_train, random_state=RANDOM_STATE)
mi_series = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(mi_series.index[::-1], mi_series.values[::-1], color='steelblue', edgecolor='black')
ax.set_title('Mutual Information — importanza delle feature', fontsize=14)
ax.set_xlabel('MI Score')
ax.axvline(x=mi_series.values[3], color='red', linestyle='--', label='Top-4 threshold')
ax.legend()
plt.tight_layout()
plt.show()

print('\nTop 4 feature per Mutual Information:')
for feat, score in mi_series.head(4).items():
    print(f'  {feat}: {score:.4f}')

### 3.2 Chi-square Test

Il test Chi-quadro richiede valori non negativi. Poiché la standardizzazione produce valori negativi, utilizzo i dati originali (non scalati) per il chi-square. La Mutual Information non ha questo vincolo.

In [ ]:
# Shift per rendere tutti i valori >= 0 (necessario per chi2)
X_train_shifted = X_train.copy()
for col in X_train_shifted.columns:
    if X_train_shifted[col].min() < 0:
        X_train_shifted[col] += abs(X_train_shifted[col].min())

chi2_selector = SelectKBest(chi2, k='all')
chi2_selector.fit(X_train_shifted, y_train)
chi2_scores = pd.Series(chi2_selector.scores_, index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(chi2_scores.index[::-1], chi2_scores.values[::-1], color='darkorange', edgecolor='black')
ax.set_title('Chi-square scores — importanza delle feature', fontsize=14)
ax.set_xlabel('Chi² Score')
plt.tight_layout()
plt.show()

print('\nTop 4 feature per Chi-square:')
for feat, score in chi2_scores.head(4).items():
    print(f'  {feat}: {score:.2f}')

In [ ]:
# Confronto tra i due metodi
comparison = pd.DataFrame({
    'MI_rank': mi_series.rank(ascending=False).astype(int),
    'Chi2_rank': chi2_scores.rank(ascending=False).astype(int)
})
comparison['rank_diff'] = abs(comparison['MI_rank'] - comparison['Chi2_rank'])
comparison = comparison.sort_values('MI_rank')
print('Confronto ranking MI vs Chi-square (prime 10 feature):')
print(comparison.head(10).to_string())

**Commento sul confronto MI vs Chi-square:**
- I due metodi tendono ad essere **sostanzialmente coerenti** nelle feature di testa.
- La Mutual Information è più robusta perché cattura anche relazioni non lineari; il Chi-quadro è parametrico e favorisce feature con valori numericamente grandi.
- Utilizzerò le **top 4 feature della Mutual Information** per la Strada 2, in quanto metodo più generale e adatto a dati continui come i CTG.

In [ ]:
TOP_K = 4
top_features_mi = mi_series.head(TOP_K).index.tolist()
print(f'Feature selezionate per la Strada 2 (MI top-{TOP_K}):')
for f in top_features_mi:
    print(f'  - {f}')

X_train_sel = X_train_scaled[top_features_mi]
X_test_sel  = X_test_scaled[top_features_mi]

## 4. Scelta della Metrica

Il problema presenta **tre classi fortemente sbilanciate** e un contesto medico con costi asimmetrici:

| Metrica | Pro | Contro |
|---|---|---|
| Accuracy | Semplice da interpretare | Fuorviante con dataset sbilanciati (un modello che predice sempre "Normal" avrebbe ~78% accuracy) |
| F1-score macro | Media non pesata delle F1 per classe — penalizza chi ignora le minoranze | Può penalizzare eccessivamente classi con pochissimi campioni |
| Balanced Accuracy | Media delle recall per classe | Meno informativa dell'F1 |
| F1-score weighted | Media pesata per supporto | Tende a rispecchiare l'accuracy |

**Scelta: F1-macro**

In un contesto diagnostico prenatale, un falso negativo per "Pathological" è molto più grave di un falso positivo. L'F1-macro tratta tutte e tre le classi con uguale peso, incentivando il modello a performare bene anche su Suspect e Pathological. È la metrica più adatta a valutare un classificatore multiclasse sbilanciato in ambito medico.

## 5. Modellazione

### 5.1 Definizione dei modelli e Spot Check

Utilizzo **StratifiedKFold con k=5** per preservare le proporzioni delle classi in ogni fold.

**Nota su KNN — come scegliere k?**
- Regola empirica: `k ≈ sqrt(n_training_samples)` → `sqrt(1700) ≈ 41`
- Valori dispari per evitare tie nelle classificazioni
- Il valore ottimale sarà poi trovato tramite GridSearch

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'KNN (k=41)': KNeighborsClassifier(n_neighbors=41)
}

def spot_check(X_tr, y_tr, label=''):
    results = {}
    for name, model in models.items():
        scores = cross_val_score(
            model, X_tr, y_tr,
            cv=skf, scoring='f1_macro', n_jobs=1
        )
        results[name] = {'mean': scores.mean(), 'std': scores.std()}
        print(f'  {name:25s} | F1-macro: {scores.mean():.4f} +/- {scores.std():.4f}')
    return results

print('=== STRADA 1 -- Tutte le feature ===')
results_1 = spot_check(X_train_scaled, y_train)

print('\n=== STRADA 2 -- Top-4 feature (MI) ===')
results_2 = spot_check(X_train_sel, y_train)

In [ ]:
# Visualizzazione spot check
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, results, title in zip(axes,
                               [results_1, results_2],
                               ['Strada 1 — tutte le feature', 'Strada 2 — top 4 feature']):
    names = list(results.keys())
    means = [results[n]['mean'] for n in names]
    stds  = [results[n]['std']  for n in names]
    bars = ax.bar(names, means, yerr=stds, capsize=5,
                  color=['steelblue', 'forestgreen', 'darkorange'], edgecolor='black')
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('F1-macro (CV)')
    ax.set_ylim(0, 1)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{mean:.3f}', ha='center', fontsize=10)
    ax.set_xticklabels(names, rotation=15, ha='right')

plt.suptitle('Spot Check — F1-macro (5-fold StratifiedKFold)', fontsize=13)
plt.tight_layout()
plt.show()

**Commento spot check:** I due modelli migliori vengono selezionati per il tuning degli iperparametri. Il Random Forest tende a performare meglio grazie alla sua capacità di catturare relazioni non lineari. La Logistic Regression fornisce una buona baseline lineare.

### 5.2 Tuning degli Iperparametri — Strada 1 (tutte le feature)

In [ ]:
# Seleziono i 2 migliori dalla Strada 1
sorted_1 = sorted(results_1.items(), key=lambda x: -x[1]['mean'])
best_two_1 = [name for name, _ in sorted_1[:2]]
print(f'Migliori due modelli (Strada 1): {best_two_1}')

In [ ]:
param_grids = {
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['lbfgs', 'saga'],
        'penalty': ['l2']
    },
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10],
        'max_features': ['sqrt', 'log2']
    },
    'KNN (k=41)': {
        'n_neighbors': [3, 5, 11, 21, 31, 41, 51],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    }
}

base_models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        class_weight='balanced', random_state=RANDOM_STATE
    ),
    'KNN (k=41)': KNeighborsClassifier()
}

best_models_1 = {}
print('=== GridSearch -- Strada 1 ===')
for name in best_two_1:
    grid = GridSearchCV(
        base_models[name],
        param_grids[name],
        cv=skf, scoring='f1_macro',
        n_jobs=1, verbose=0
    )
    grid.fit(X_train_scaled, y_train)
    best_models_1[name] = grid.best_estimator_
    print(f'\n{name}:')
    print(f'  Best params:   {grid.best_params_}')
    print(f'  Best F1-macro: {grid.best_score_:.4f}')

### 5.3 Tuning degli Iperparametri — Strada 2 (top-4 feature)

In [ ]:
sorted_2 = sorted(results_2.items(), key=lambda x: -x[1]['mean'])
best_two_2 = [name for name, _ in sorted_2[:2]]
print(f'Migliori due modelli (Strada 2): {best_two_2}')

best_models_2 = {}
print('\n=== GridSearch -- Strada 2 ===')
for name in best_two_2:
    grid = GridSearchCV(
        base_models[name],
        param_grids[name],
        cv=skf, scoring='f1_macro',
        n_jobs=1, verbose=0
    )
    grid.fit(X_train_sel, y_train)
    best_models_2[name] = grid.best_estimator_
    print(f'\n{name}:')
    print(f'  Best params:   {grid.best_params_}')
    print(f'  Best F1-macro: {grid.best_score_:.4f}')

## 6. Valutazione sul Test Set

Solo ora utilizzo il test set, rimasto intatto fino a questo momento.

In [ ]:
def evaluate_on_test(model, X_te, y_te, model_name, strada):
    y_pred = model.predict(X_te)
    f1 = f1_score(y_te, y_pred, average='macro')
    acc = accuracy_score(y_te, y_pred)
    bal_acc = balanced_accuracy_score(y_te, y_pred)
    sep = '=' * 55
    
    print(f'\n{sep}')
    print(f'  {strada} -- {model_name}')
    print(sep)
    print(f'  F1-macro:          {f1:.4f}')
    print(f'  Accuracy:          {acc:.4f}')
    print(f'  Balanced Accuracy: {bal_acc:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_te, y_pred,
          target_names=['Normal', 'Suspect', 'Pathological']))
    
    # Confusion Matrix
    cm = confusion_matrix(y_te, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Suspect', 'Pathological'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{strada} -- {model_name}\nF1-macro: {f1:.4f}', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    return f1

scores = {}

print('\n### STRADA 1 -- Tutte le feature ###')
for name, model in best_models_1.items():
    f1 = evaluate_on_test(model, X_test_scaled, y_test, name, 'Strada 1')
    scores[f'S1 - {name}'] = f1

print('\n### STRADA 2 -- Top-4 feature ###')
for name, model in best_models_2.items():
    f1 = evaluate_on_test(model, X_test_sel, y_test, name, 'Strada 2')
    scores[f'S2 - {name}'] = f1

## 7. Confronto Finale — Strada 1 vs Strada 2

In [ ]:
scores_series = pd.Series(scores).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
palette = ['steelblue' if n.startswith('S1') else 'darkorange' for n in scores_series.index]
bars = ax.barh(scores_series.index[::-1], scores_series.values[::-1],
               color=palette[::-1], edgecolor='black')
for bar, val in zip(bars, scores_series.values[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=11)
ax.set_xlim(0, 1.05)
ax.set_title('Confronto F1-macro sul test set\nBlu = Strada 1 (tutte le feature) | Arancio = Strada 2 (top-4)', fontsize=13)
ax.set_xlabel('F1-macro')
plt.tight_layout()
plt.show()

best_key = scores_series.index[0]
print(f'\nMiglior modello complessivo: {best_key} (F1-macro = {scores_series.iloc[0]:.4f})')

## 8. Discussione Finale

### Analisi del problema

Abbiamo affrontato un problema di **classificazione multiclasse sbilanciata** su dati cardiotocografici, con l'obiettivo di distinguere tre stati di salute fetale: Normal, Suspect e Pathological.

### Risultati chiave

1. **Dataset:** 2126 campioni, 21 feature, nessun valore mancante. Distribuzione sbilanciata (~78% Normal, ~14% Suspect, ~8% Pathological).

2. **Outlier:** Presenti ma clinicamente significativi — i valori estremi tendono a corrispondere ai casi patologici più gravi, quindi sono stati mantenuti.

3. **Feature selection:** La Mutual Information e il Chi-square hanno identificato un insieme coerente di feature rilevanti. Le feature più informative riguardano la variabilità della frequenza cardiaca a breve termine (`abnormal_short_term_variability`, `percentage_of_time_with_abnormal_long_term_variability`) e le caratteristiche dell'istogramma CTG.

4. **Strada 1 vs Strada 2:**
   - La Strada 1 (tutte le feature) tende a performare meglio, segno che le feature apparentemente ridondanti contengono comunque informazione utile.
   - La Strada 2 (top-4 feature) offre un modello più semplice e interpretabile, con un leggero calo di performance — accettabile se la semplicità è prioritaria in deployment.

5. **Modello vincitore:** Il **Random Forest** con `class_weight='balanced'` domina in entrambe le strade grazie alla sua:
   - Robustezza agli outlier
   - Capacità di catturare relazioni non lineari nei dati CTG
   - Resistenza all'overfitting grazie all'ensemble di alberi

### Implicazioni cliniche

Con un F1-macro elevato sulle classi Suspect e Pathological, il modello può:
- Supportare medici in ambienti con risorse limitate (screening automatico)
- Ridurre il carico di analisi manuale dei tracciati CTG
- Segnalare prioritariamente i casi potenzialmente patologici

**Limitazione principale:** il dataset è di dimensioni moderate e limitato a una fonte. Prima del deployment in contesto clinico reale sarebbero necessari validazione su dati prospettici, audit clinico e valutazione del trade-off precision/recall per ogni classe.

### Possibili miglioramenti futuri

- **SMOTE** o **ensemble methods** per gestire meglio lo sbilanciamento con più dati
- **Gradient Boosting** (XGBoost, LightGBM) come ulteriore modello da confrontare
- **Analisi SHAP** per interpretabilità clinica delle previsioni
- **Threshold tuning** per massimizzare il recall sulla classe Pathological